In [13]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = 'Magalong'
STUDENT_ID = 'TUPM-26-1134'

id_digits = ''.join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, 'Student ID must contain digits.'

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print('Student:', LAST_NAME, STUDENT_ID)
print('Seed Digit:', SEED_DIGIT)
print('ID Sum:', ID_SUM)
print('Name Length:', NAME_LENGTH)
print('Random Seed:', RANDOM_SEED)
print('Verification Value:', verification_value)

Student: Magalong TUPM-26-1134
Seed Digit: 4
ID Sum: 17
Name Length: 8
Random Seed: 261134
Verification Value: 140


In [14]:
# [CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / 'data'
OUTPUT_DIR = WORK_DIR / 'output'

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / 'secom.data'
label_path = DATA_DIR / 'secom_labels.data'
description_path = DATA_DIR / 'secom.names'

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f'Missing file: {path}'

print('Workspace:', WORK_DIR.resolve())
print('All dataset files verified.')

Workspace: C:\Users\Paul\Documents\TUPM-BSECE\1C\AECEL1\Lab\Lab7_NumPy_Pandas_Magalong\Lab7_Additionals
All dataset files verified.


In [15]:
# [CELL 3] CONSTRUCT LABELED DATAFRAME
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw_measurements.ndim == 2
assert len(labels) == len(raw_measurements)
assert np.isin(labels, [-1, 1]).all()

n_features = raw_measurements.shape[1]
feature_names = [f'Feature_{i:03d}' for i in range(n_features)]

features_df = pd.DataFrame(raw_measurements, columns=feature_names)

meta_df = pd.DataFrame({
    'Example_ID': np.arange(len(labels)),
    'Result': labels,
    'Outcome': np.where(labels == -1, 'Pass', 'Fail')
})

df = pd.concat([meta_df, features_df], axis=1)
assert df['Example_ID'].is_unique

print('DataFrame Shape:', df.shape)
print('Pass Records:', (df['Outcome'] == 'Pass').sum())
print('Fail Records:', (df['Outcome'] == 'Fail').sum())
print(df[['Example_ID', 'Result', 'Outcome']].head())

DataFrame Shape: (1567, 593)
Pass Records: 1463
Fail Records: 104
   Example_ID  Result Outcome
0           0      -1    Pass
1           1      -1    Pass
2           2       1    Fail
3           3      -1    Pass
4           4      -1    Pass


In [16]:
# [CELL 4] IDENTIFY USABLE FEATURES
observed_counts = features_df.notna().sum()
distinct_values = features_df.nunique()

usable_features = [
    name for name in feature_names
    if observed_counts[name] > 0 and distinct_values[name] > 1
]

print('Entirely Missing Features:', (observed_counts == 0).sum())
print('Constant Features:', ((observed_counts > 0) & (distinct_values == 1)).sum())
print('Usable Features:', len(usable_features))

Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474


In [17]:
# [CELL 5] REPRODUCIBLE BALANCED SAMPLE
pass_count = (df['Outcome'] == 'Pass').sum()
fail_count = (df['Outcome'] == 'Fail').sum()

per_group = min(20 + ID_SUM, pass_count, fail_count)

balanced = pd.concat([
    df[df['Outcome'] == 'Pass'].sample(n=per_group, random_state=RANDOM_SEED),
    df[df['Outcome'] == 'Fail'].sample(n=per_group, random_state=RANDOM_SEED)
])

balanced = balanced.sort_values('Example_ID').reset_index(drop=True)

print('Records per Outcome:', per_group)
print('Balanced Sample Size:', len(balanced))
print(balanced['Outcome'].value_counts())

Records per Outcome: 37
Balanced Sample Size: 74
Outcome
Fail    37
Pass    37
Name: count, dtype: int64


In [18]:
# [CELL 6] SELECT FEATURES AND APPLY MEDIAN IMPUTATION
rng = np.random.default_rng(RANDOM_SEED)
positions = rng.choice(len(usable_features), size=5, replace=False)

selected_features = [usable_features[i] for i in positions]

# Medians come from the FULL dataset, not from the balanced sample
full_dataset_medians = df[selected_features].median()

balanced_sample = balanced[
    ['Example_ID'] + selected_features + ['Result', 'Outcome']
].copy()

missing_before = balanced_sample[selected_features].isna().sum().sum()
balanced_sample[selected_features] = balanced_sample[selected_features].fillna(
    full_dataset_medians
)
missing_after = balanced_sample[selected_features].isna().sum().sum()

print('Selected Features:', selected_features)
print('Full-Dataset Medians:\n', full_dataset_medians.round(4))
print('Missing Before Imputation:', missing_before)
print('Missing After Imputation:', missing_after)

Selected Features: ['Feature_523', 'Feature_215', 'Feature_287', 'Feature_029', 'Feature_345']
Full-Dataset Medians:
 Feature_523    0.1000
Feature_215    0.0825
Feature_287    0.1390
Feature_029    2.3778
Feature_345    5.5670
dtype: float64
Missing Before Imputation: 32
Missing After Imputation: 0


In [19]:
# [CELL 7] GROUPED STATISTICS USING GROUPBY AND AGG
grouped = balanced_sample.groupby('Outcome')[selected_features].agg(
    ['count', 'mean', 'median', 'std']
)

print(grouped.round(4))

        Feature_523                         Feature_215                  \
              count    mean  median     std       count    mean  median   
Outcome                                                                   
Fail             37  0.3173  0.0805  0.6984          37  0.0853  0.0840   
Pass             37  0.4943  0.0972  1.0042          37  0.0683  0.0737   

                Feature_287                         Feature_029          \
            std       count    mean  median     std       count    mean   
Outcome                                                                   
Fail     0.0327          37  0.2332  0.1592  0.2172          37  2.3258   
Pass     0.0253          37  0.1830  0.1415  0.0885          37  2.4216   

                        Feature_345                         
         median     std       count    mean median     std  
Outcome                                                     
Fail     2.3667  0.4139          37  5.8201  5.567  1.7856  
Pass

In [20]:
# [CELL 8] STANDARDIZED MEAN DIFFERENCES AND RANKING

rows = []

for feature in selected_features:
    n_pass, n_fail = grouped.loc['Pass', (feature, 'count')], grouped.loc['Fail', (feature, 'count')]
    mean_pass, mean_fail = grouped.loc['Pass', (feature, 'mean')], grouped.loc['Fail', (feature, 'mean')]
    med_pass, med_fail = grouped.loc['Pass', (feature, 'median')], grouped.loc['Fail', (feature, 'median')]
    sd_pass, sd_fail = grouped.loc['Pass', (feature, 'std')], grouped.loc['Fail', (feature, 'std')]

    dof = n_pass + n_fail - 2
    if dof > 0 and np.isfinite(sd_pass) and np.isfinite(sd_fail):
        pooled_sd = np.sqrt(((n_pass - 1) * sd_pass**2 + (n_fail - 1) * sd_fail**2) / dof)
    else:
        pooled_sd = np.nan

    mean_difference = mean_fail - mean_pass
    zero_denominator = (not np.isfinite(pooled_sd)) or pooled_sd == 0
    smd = 0.0 if zero_denominator else mean_difference / pooled_sd

    rows.append({
        'Feature': feature,
        'Pass_Count': int(n_pass),
        'Fail_Count': int(n_fail),
        'Pass_Mean': mean_pass,
        'Fail_Mean': mean_fail,
        'Pass_Median': med_pass,
        'Fail_Median': med_fail,
        'Pass_SD': sd_pass,
        'Fail_SD': sd_fail,
        'Abs_Mean_Difference': abs(mean_difference),
        'Pooled_SD': pooled_sd,
        'Standardized_Mean_Difference': smd,
        'Abs_Standardized_Mean_Difference': abs(smd),
        'Zero_Denominator': zero_denominator,
    })

comparison = (
    pd.DataFrame(rows)
    .sort_values('Abs_Standardized_Mean_Difference', ascending=False, kind='stable')
    .reset_index(drop=True)
)
comparison.insert(0, 'Rank', np.arange(1, len(comparison) + 1))

print(comparison[['Rank', 'Feature', 'Abs_Mean_Difference',
                  'Pooled_SD', 'Standardized_Mean_Difference']].round(4))

   Rank      Feature  Abs_Mean_Difference  Pooled_SD  \
0     1  Feature_215               0.0170     0.0293   
1     2  Feature_287               0.0502     0.1659   
2     3  Feature_029               0.0958     0.4418   
3     4  Feature_523               0.1771     0.8649   
4     5  Feature_345               0.0986     1.5986   

   Standardized_Mean_Difference  
0                        0.5809  
1                        0.3029  
2                       -0.2169  
3                       -0.2047  
4                       -0.0617  


In [21]:
# [CELL 9] EXPORT REQUIRED OUTPUTS
sample_path = OUTPUT_DIR / 'balanced_sample.csv'
comparison_path = OUTPUT_DIR / 'feature_comparison.csv'

balanced_sample.to_csv(sample_path, index=False)
comparison.to_csv(comparison_path, index=False)

for path in [sample_path, comparison_path]:
    print(path.name, 'Created:', path.is_file())

balanced_sample.csv Created: True
feature_comparison.csv Created: True


In [22]:
# [CELL 10] NUMPY-PANDAS VERIFICATION
pass_values = balanced_sample.loc[
    balanced_sample['Outcome'] == 'Pass', selected_features
].to_numpy()
fail_values = balanced_sample.loc[
    balanced_sample['Outcome'] == 'Fail', selected_features
].to_numpy()

numpy_pass_means = pass_values.mean(axis=0)
numpy_fail_means = fail_values.mean(axis=0)
numpy_pass_sds = pass_values.std(axis=0, ddof=1)
numpy_fail_sds = fail_values.std(axis=0, ddof=1)

pandas_pass_means = np.array([grouped.loc['Pass', (f, 'mean')] for f in selected_features])
pandas_fail_means = np.array([grouped.loc['Fail', (f, 'mean')] for f in selected_features])
pandas_pass_sds = np.array([grouped.loc['Pass', (f, 'std')] for f in selected_features])
pandas_fail_sds = np.array([grouped.loc['Fail', (f, 'std')] for f in selected_features])

stats_match = (
    np.allclose(numpy_pass_means, pandas_pass_means)
    and np.allclose(numpy_fail_means, pandas_fail_means)
    and np.allclose(numpy_pass_sds, pandas_pass_sds)
    and np.allclose(numpy_fail_sds, pandas_fail_sds)
)

# Confirm the exported sample keeps the original record IDs and measurements
loaded_sample = pd.read_csv(sample_path)
original_rows = df.set_index('Example_ID').loc[loaded_sample['Example_ID']]

ids_match = np.array_equal(loaded_sample['Result'], original_rows['Result'])
export_match = len(loaded_sample) == len(balanced_sample)

print('NumPy-Pandas Statistics Match:', stats_match)
print('Original Record IDs Preserved:', ids_match)
print('Exported Records Match:', export_match)

NumPy-Pandas Statistics Match: True
Original Record IDs Preserved: True
Exported Records Match: True


In [23]:
# [CELL 11] FINAL OUTPUT
best = comparison.iloc[0]

print('ASSESSMENT DATA - PANDAS OUTCOME COMPARISON')
print('-------------------------------------------')
print('Balanced Sample Size:', len(balanced_sample),
      f'({per_group} Pass + {per_group} Fail)')
print('Selected Features:', ', '.join(selected_features))
print('Highest-Ranked Feature:', best['Feature'])
print('Absolute Standardized Mean Difference:',
      f"{best['Abs_Standardized_Mean_Difference']:.4f}")
print('NumPy-Pandas Statistics Match:', stats_match)
print('Required Outputs:', sample_path.name, ',', comparison_path.name)

ASSESSMENT DATA - PANDAS OUTCOME COMPARISON
-------------------------------------------
Balanced Sample Size: 74 (37 Pass + 37 Fail)
Selected Features: Feature_523, Feature_215, Feature_287, Feature_029, Feature_345
Highest-Ranked Feature: Feature_215
Absolute Standardized Mean Difference: 0.5809
NumPy-Pandas Statistics Match: True
Required Outputs: balanced_sample.csv , feature_comparison.csv
